# DGCA domestic airlines: interactive Plotly Dash dashboard
Fourth notebook. Reads the cleaned table  and serves the dashboard as a **Plotly Dash app**. Every chart is a Plotly figure inside a , and the page (header cards, tabs, airline picker, captions, methodology) is built from Dash components, so Dash generates the HTML you see in the browser.

What the page contains:
- four headline cards, tabs for the figures (including capacity growth and a playable route-density map of India) plus a methodology tab, and an **airline picker** (shown on the two line-chart tabs) that updates a summary and highlights that airline,
- Dash controls instead of Plotly buttons: radio buttons switch views and zoom periods, and a slider with a Play/Pause button runs the month-by-month race,
- a caption with numbers under every figure, a "How we counted" section, and sources.

Numbers are stated in everyday units: passengers per month in millions, "out of every 100 passengers", "out of every 100 seats", and the same month of 2019 as the yardstick.

In [ ]:
%pip install -q dash plotly pyarrow pandas numpy

## 1. Setup and data
Group-level monthly series for scheduled domestic flights, cut at the last month where the airlines add up to DGCA's own total.

In [ ]:
# 1. SETUP AND DATA: group-level monthly series for scheduled domestic traffic (source: data/clean/dgca_domestic_clean_*.parquet)
import json
from pathlib import Path
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output, State, ctx

TODAY = pd.Timestamp.today().date().isoformat()
_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "data" / "clean").is_dir())
RAW, INTERIM, CLEAN = (PROJECT_ROOT / "data" / d for d in ("raw", "interim", "clean"))
latest = lambda folder, pat: sorted(folder.glob(pat))[-1]

clean = pd.read_parquet(latest(CLEAN, "dgca_domestic_clean_*.parquet"))
interim = pd.read_csv(latest(INTERIM, "dgca_domestic_airline_monthly_*.csv"))
rec = pd.read_csv(latest(CLEAN, "total_domestic_reconciliation_*.csv"), parse_dates=["Date"])
LAST = rec.loc[rec.within_2pct, "Date"].max()                 # last month where the airlines add up to DGCA's own total
START = pd.Timestamp("2019-01-01")
HL, DARK, MID, LIGHT = "#d95f02", "#404040", "#8c8c8c", "#d0d0d0"     # one highlight colour, the rest grey
MIN_BASE_SHARE = 0.005          # a group needs >= 0.5% of 2019 passengers for a fair 2019 comparison
FONT = "system-ui, -apple-system, 'Segoe UI', Roboto, Arial, sans-serif"

fm = lambda d: pd.Timestamp(d).strftime("%b %Y")
iso = lambda idx: [pd.Timestamp(d).strftime("%Y-%m-%d") for d in idx]
lst = lambda s, k=4: [None if pd.isna(v) else round(float(v), k) for v in s]

sdc = clean[(clean.Type == "ScheduledDomestic") & (clean.Date <= LAST)]
months = pd.date_range("2017-01-01", LAST, freq="MS")
piv = lambda col: sdc.groupby(["Group", "Date"])[col].sum(min_count=1).unstack("Group").reindex(months)
pax, pkp, skp = piv("Passenger Number"), piv("Passenger Kilometers"), piv("Seat Kilometers")

# A month with no report is counted as zero passengers only where the cleaning notebook (BLOCK 8b) found evidence of no flying:
# DGCA reported zero, nobody reported anything that month, or the airlines still add up to DGCA's Total Domestic without it. Other gaps stay unknown (NaN).
status = pd.read_csv(latest(CLEAN, "airline_month_status_*.csv"), parse_dates=["Date"])
paxf = pax.copy()
for g, d in status.loc[status.treat_as_zero, ["Group", "Date"]].itertuples(index=False):
    if g in paxf.columns and d in paxf.index and pd.isna(paxf.at[d, g]):
        paxf.at[d, g] = 0
unknown_months = status[~status.treat_as_zero & status.status.isin(["published_blank", "no_source_row", "reported_value_missing"]) & (status.Date >= START)]
print("months counted as zero on evidence:", int((status.treat_as_zero & (status.status != "reported_zero")).sum()), "| missing months left unknown since 2019:", len(unknown_months))
total = paxf.sum(axis=1)

b19 = pax[pax.index.year == 2019]
pos12, share19 = (b19 > 0).sum(), b19.sum() / b19.sum().sum()
keep = [g for g in pax.columns if pos12[g] == 12 and share19[g] >= MIN_BASE_SHARE]
excluded = {g: ("no 2019 traffic" if b19[g].sum() == 0 else f"only {pos12[g]} months in 2019" if pos12[g] < 12 else "too small in 2019")
            for g in pax.columns if g not in keep and paxf[g].loc[START:].max() >= 40_000}
base = b19[keep].copy(); base.index = base.index.month
ratio = paxf[keep].div(base.reindex(paxf.index.month).set_axis(paxf.index))

# groups worth drawing: at least 40,000 passengers in some month since 2019, biggest in 2019 first
show = [g for g in paxf.columns if paxf[g].loc[START:].max() >= 40_000]
show.sort(key=lambda g: (-b19[g].sum(), -paxf[g].iloc[-3:].mean()))
big = [g for g in show if paxf[g].loc[START:].max() >= 500_000 and g != "Jet Airways"]   # drawn by default in the passenger view
print("airline groups drawn:", show); print("excluded from the 2019 comparison:", excluded)

## 2. Numbers for the cards, the airline picker and the captions

In [ ]:
# 2. NUMBERS FOR THE CARDS, THE AIRLINE PICKER AND THE CAPTIONS
HL_GROUP = "IndiGo"
n3 = slice(-3, None)
last3 = paxf.index[n3]

# market-level
tot19_by_month = paxf[paxf.index.year == 2019].sum(axis=1); tot19_by_month.index = tot19_by_month.index.month
ratio_total = total / tot19_by_month.reindex(total.index.month).values
roll_total = ratio_total.rolling(3).mean()
hit = roll_total.loc["2020-07-01":][roll_total.loc["2020-07-01":] >= 1.0]
market_back = hit.index[0] if len(hit) else None
def sustained_from(roll, n=6):
    """First month from which the 3-month average stays at or above 100% for n months in a row (the first crossing can be a one-off)."""
    above = roll.dropna().ge(1.0); run = above.astype(float)[::-1].rolling(n, min_periods=n).min()[::-1].eq(1.0)
    return above.index[run][0] if run.any() else None
market_sust = sustained_from(roll_total.loc["2020-07-01":])
avg19 = tot19_by_month.mean() / 1e6
trough_val, trough_date = total.loc[START:].min() / 1e6, total.loc[START:].idxmin()
l12 = total.iloc[-12:].mean() / 1e6
latest_m = total.index[-1]
same_m_2019 = total[(total.index.year == 2019) & (total.index.month == latest_m.month)].iloc[0] / 1e6

# per group
res_rows = {}
for g in keep:
    roll = ratio[g].rolling(3).mean().loc["2020-07-01":]; h = roll[roll >= 1.0]
    res_rows[g] = dict(recovered=h.index[0] if len(h) else None, latest_pct=ratio[g].iloc[n3].mean() * 100, sustained=sustained_from(roll))

def lf_between(g, sl):
    a, b = pkp[g].loc[sl].sum(), skp[g].loc[sl].sum()
    return a / b * 100 if b > 0 else np.nan
sh_all = paxf.div(total, axis=0) * 100
stats = {}
for g in show:
    last_tr = paxf[g][paxf[g] > 0].index.max(); first_tr = paxf[g][paxf[g] > 0].index.min()
    stopped = last_tr <= LAST - pd.Timedelta(days=62)
    now = paxf[g].iloc[n3].mean() / 1e6
    s = dict(now_m=round(now, 2), share19=round(share19.get(g, 0) * 100, 1), share_now=round(sh_all[g].iloc[n3].mean(), 1),
             lf19=None, lf_now=None, avg19_m=None, pct=None, back=None)
    if g in keep:
        s["avg19_m"] = round(b19[g].mean() / 1e6, 2); s["pct"] = round(res_rows[g]["latest_pct"])
        s["back"] = fm(res_rows[g]["recovered"]) if res_rows[g]["recovered"] is not None else "never"
    l19, lnow = lf_between(g, slice("2019-01-01", "2019-12-01")), lf_between(g, slice(paxf.index[-12], None))
    s["lf19"] = None if np.isnan(l19) else round(l19); s["lf_now"] = None if np.isnan(lnow) else round(lnow)
    if g in keep and stopped:
        s["sentence"] = f"Stopped flying after {fm(last_tr)}. It never got back to its 2019 passenger numbers."
    elif g in keep and res_rows[g]["recovered"] is not None and s["pct"] >= 100:
        sus = res_rows[g]["sustained"]; since = fm(sus) if sus is not None else s["back"]
        s["sentence"] = f"Back above its 2019 passenger numbers since {since}" + (f" (it first touched the 2019 level in {s['back']})" if sus is not None and sus != res_rows[g]["recovered"] else "") + f". Now carries {s['pct']}% of what it carried in the same months of 2019."
    elif g in keep and res_rows[g]["recovered"] is not None:
        s["sentence"] = f"Reached its 2019 level in {s['back']}, but has since slipped to {s['pct']}% of 2019."
    elif g in keep:
        s["sentence"] = f"Still flying, but carries only {s['pct']}% of what it carried in 2019. It has never got back to 2019."
    elif stopped:
        s["sentence"] = f"Stopped flying after {fm(last_tr)} (it had no full 2019 year to compare with)."
    elif b19[g].sum() == 0:
        s["sentence"] = f"Started flying after 2019 (first passengers: {fm(first_tr)}), so there is no 2019 level to compare with. Now carries {now:.2f} million passengers a month."
    else:
        s["sentence"] = "Too small in 2019 for a fair comparison with 2019."
    stats[g] = s

# cards
big_back = [g for g in keep if res_rows[g]["latest_pct"] >= 100]
top = max(show, key=lambda g: sh_all[g].iloc[n3].mean())
CARDS = [
    (f"{total.iloc[-1] / 1e6:.1f} million", f"passengers flew in {latest_m:%B %Y}", f"{same_m_2019:.1f} million in {latest_m:%B} 2019 ({(total.iloc[-1] / 1e6 / same_m_2019 - 1) * 100:+.0f}%)"),
    (fm(market_back) if market_back is not None else "not yet", "market back to its 2019 level", "first month the 3-month average matched the same months of 2019" + ("" if market_sust is None or market_sust == market_back else f"; it has stayed there since {fm(market_sust)}")),
    (f"{sh_all[top].iloc[n3].mean():.0f} in 100", f"passengers fly {top} today", f"it was {share19.get(top, 0) * 100:.0f} in 100 in 2019"),
    (f"{len(big_back)} of {len(keep)}", "airline groups that flew all of 2019 are back above it", ", ".join(big_back) if big_back else "none"),
]
print(CARDS); print({g: stats[g]["sentence"] for g in ["IndiGo", "SpiceJet", "Go First", "Akasa Air"] if g in stats})

## 2b. Capacity and route data
Seat-kilometres for the capacity tab, and city-pair passengers for the route density tab. The city-pair table is built by section 9 of the download notebook (`data/interim/dgca_domestic_citypair_monthly_*.csv`).

In [ ]:
# 2b. DATA FOR THE CAPACITY TAB AND THE ROUTE DENSITY TAB
import re

# ---- capacity: seat-kilometres (the airline's seats multiplied by distance flown, i.e. what it put on sale)
skf = skp.copy()                # same assumption as for passengers: a group that stopped flying has 0 capacity
for g in skf.columns:
    f = skf[g].first_valid_index()
    if f is not None:
        skf.loc[(skf.index >= f) & skf[g].isna(), g] = 0
sk19 = skp[skp.index.year == 2019].copy(); sk19.index = sk19.index.month
cap_ok = [g for g in keep if (sk19[g] > 0).all()]                       # a full, positive 2019 capacity base
cap_ratio = skf[cap_ok].div(sk19[cap_ok].reindex(skf.index.month).set_axis(skf.index)) * 100
mkt_sk = skf.sum(axis=1)
mkt_sk19 = mkt_sk[mkt_sk.index.year == 2019]; mkt_sk19.index = mkt_sk19.index.month
cap_total_now = mkt_sk.iloc[n3].mean() / mkt_sk19.reindex(mkt_sk.index[n3].month).mean() * 100
print("capacity groups:", cap_ok, "| market capacity now vs 2019: %.0f%%" % cap_total_now)

# ---- route density: monthly passengers between city pairs, from the download pipeline's combined table (data/interim/dgca_domestic_citypair_monthly_*.csv)
CITY_FILE = latest(INTERIM, "dgca_domestic_citypair_monthly_*.csv")
city = pd.read_csv(CITY_FILE)
SUPPLEMENT_MONTHS = sorted({f"{pd.Timestamp(year=int(y), month=int(m), day=1):%b %Y}" for y, m, src in city[["Year", "Month", "Source"]].drop_duplicates().itertuples(index=False) if src != "vonter"},
                           key=lambda t: pd.Timestamp(t))       # months taken from DGCA's own files because the Vonter table lacks them
def city_name(s):
    s = re.sub(r"\s*\(.*\)$", "", str(s).strip().upper())               # "MUMBAI (NAVI MUMBAI)" -> "MUMBAI"
    return "GOA" if s == "DABOLIM" else s                                # both Goa airports are one city on the map
city["Date"] = pd.to_datetime(dict(year=city.Year, month=city.Month, day=1))
a, b = city.City1.map(city_name), city.City2.map(city_name)
city["A"], city["B"] = np.where(a < b, a, b), np.where(a < b, b, a)       # one row per pair, whichever way it is written
city["pax"] = city.PaxToCity2.fillna(0) + city.PaxFromCity2.fillna(0)     # passengers in both directions
pairs = city[city.A != city.B].groupby(["Date", "A", "B"], as_index=False)["pax"].sum()
RT_MONTHS = pd.date_range(START, min(LAST, pairs.Date.max()), freq="MS")
pairs = pairs[pairs.Date.isin(RT_MONTHS)]
cvol = pd.concat([pairs.rename(columns={"A": "City"})[["Date", "City", "pax"]], pairs.rename(columns={"B": "City"})[["Date", "City", "pax"]]]).groupby(["Date", "City"])["pax"].sum().unstack("City").reindex(RT_MONTHS).fillna(0)

COORDS = {"DELHI": (28.61, 77.21), "MUMBAI": (19.08, 72.88), "BENGALURU": (12.97, 77.59), "HYDERABAD": (17.38, 78.48), "KOLKATA": (22.57, 88.36),
          "CHENNAI": (13.08, 80.27), "AHMEDABAD": (23.02, 72.57), "PUNE": (18.52, 73.86), "GUWAHATI": (26.14, 91.74), "KOCHI": (9.93, 76.27),
          "LUCKNOW": (26.85, 80.95), "GOA": (15.38, 73.83), "JAIPUR": (26.91, 75.79), "BHUBANESWAR": (20.30, 85.82), "PATNA": (25.59, 85.14),
          "SRINAGAR": (34.08, 74.80), "INDORE": (22.72, 75.86), "CHANDIGARH": (30.73, 76.78), "VARANASI": (25.32, 83.01), "BAGDOGRA": (26.68, 88.33),
          "COIMBATORE": (11.02, 76.96), "NAGPUR": (21.15, 79.09), "VISAKHAPATNAM": (17.69, 83.22), "RANCHI": (23.34, 85.31), "RAIPUR": (21.25, 81.63)}
MAJOR = ["DELHI", "MUMBAI", "BENGALURU", "HYDERABAD", "KOLKATA", "CHENNAI"]          # the six metros shown on the map
N_CITIES = len(MAJOR); N_ROUTES = N_CITIES * (N_CITIES - 1) // 2                  # every pair of them: 15 routes
print("cities on the map:", MAJOR, "| routes:", N_ROUTES)
major_pairs = pairs[pairs.A.isin(MAJOR) & pairs.B.isin(MAJOR)]
PAIR_MAX = major_pairs.pax.max(); CITY_MAX = cvol[MAJOR].max().max()          # fixed scales, so line widths and dot sizes are comparable across months
pairs_by_month = {d: major_pairs[major_pairs.Date == d] for d in RT_MONTHS}
RT_MISSING = [d for d in RT_MONTHS if d not in set(pairs.Date)]          # months with no city-pair file (DGCA did not publish them)
print("months with no city-pair data:", [fm(d) for d in RT_MISSING])
print("route months:", fm(RT_MONTHS[0]), "to", fm(RT_MONTHS[-1]), "| busiest pair:", major_pairs.loc[major_pairs.pax.idxmax(), ["A", "B", "pax"]].tolist())
chk = pairs.groupby("Date").pax.sum() / total.reindex(RT_MONTHS).astype(float)
print("city-pair total vs airline total (ratio, should be about 1):", chk.min().round(3), "-", chk.max().round(3))

## 3. The figures
Each figure is a function. Dash callbacks call it again when a control (picker, radio buttons, slider) changes.

In [ ]:
# 3. THE SEVEN FIGURES. Each is a function, so a Dash callback can rebuild it when a control changes.
CFG = dict(displaylogo=False, responsive=True, modeBarButtonsToRemove=["lasso2d", "select2d", "autoScale2d"])
def style(fig, h=470, legend=True):
    fig.update_layout(template="plotly_white", font=dict(family=FONT, size=13, color="#222"), height=h,
                      margin=dict(l=64, r=40, t=36, b=60), hoverlabel=dict(font_size=13),
                      legend=dict(orientation="h", y=-0.22, x=0) if legend else None, showlegend=legend)
    return fig
WAVES = [("2020-03-01", "2020-06-01", "lockdown"), ("2021-04-01", "2021-07-01", "2nd wave")]
def waves(fig):
    for a, b, lab in WAVES:
        fig.add_vrect(x0=a, x1=b, fillcolor=LIGHT, opacity=.45, layer="below", line_width=0)
        fig.add_annotation(x=a, y=1.0, xref="x", yref="paper", text=lab, showarrow=False, xanchor="left", yanchor="bottom", font=dict(size=11, color=MID))
DATES = iso(paxf.loc[START:].index)
mon = lambda d: pd.Timestamp(d).strftime("%b %Y")
b19m = pax[pax.index.year == 2019].copy(); b19m.index = b19m.index.month
X_END = iso([LAST + pd.DateOffset(months=9)])[0]          # room on the right for the line labels

# ---- 1. the whole market
MARKET_RANGES = {"all": [DATES[0], DATES[-1]], "2019-21": ["2019-01-01", "2021-12-31"], "2022+": ["2022-01-01", DATES[-1]]}
def build_market(rng="all"):
    t = total.loc[START:] / 1e6
    f = go.Figure(go.Scatter(x=DATES, y=lst(t, 3), mode="lines", line=dict(color=DARK, width=3), name="Passengers",
            hovertext=[f"<b>{mon(d)}</b><br>{v:.1f} million passengers<br>{v / avg19 * 100:.0f}% of the 2019 monthly average" for d, v in zip(t.index, t)], hoverinfo="text"))
    f.add_hline(y=avg19, line_dash="dash", line_color="black", line_width=1, annotation_text=f"2019 average: {avg19:.1f} million a month", annotation_position="top left", annotation_font_size=12)
    f.add_trace(go.Scatter(x=[iso([trough_date])[0]], y=[trough_val], mode="markers+text", marker=dict(color=HL, size=13), text=[f"{mon(trough_date)}: {trough_val:.1f} million"],
            textposition="top right", hoverinfo="skip", name="Lowest month", showlegend=False))
    waves(f)
    f.update_layout(xaxis=dict(rangeslider=dict(visible=True, thickness=.07), range=MARKET_RANGES[rng]),
                    yaxis=dict(title="Passengers per month (millions)", rangemode="tozero"))
    return style(f, 440, legend=False)

# ---- 2. airline by airline (two views: passengers, or % of the same month of 2019); `hl` is the picked group
def line(g, x, y, hover, visible, showlegend, hl):
    on = g == hl
    return go.Scatter(x=x, y=y, mode="lines+text", name=g, visible=visible, showlegend=showlegend, hovertext=hover, hoverinfo="text",
                      line=dict(color=HL if on else MID, width=4 if on else (1.6 if hl else 2.2)), opacity=1 if (on or not hl) else .55,
                      text=[""] * (len(y) - 1) + [g], textposition="middle right", textfont=dict(color=HL if on else MID, size=11), connectgaps=False)
HV_PAX, HV_PCT = {}, {}
for g in show:
    v = paxf[g].loc[START:] / 1e6
    HV_PAX[g] = [f"<b>{g}</b><br>{mon(d)}: {x:.2f} million passengers" + (f"<br>{ratio[g].loc[d] * 100:.0f}% of {pd.Timestamp(d):%B} 2019" if g in keep else "") for d, x in zip(v.index, v)]
for g in keep:
    r = ratio[g].loc[START:] * 100
    HV_PCT[g] = [f"<b>{g}</b><br>{mon(d)}: {x:.0f}% of {pd.Timestamp(d):%B} 2019<br>({paxf[g].loc[d] / 1e6:.2f} million passengers vs {b19m[g].loc[d.month] / 1e6:.2f} million in 2019)" for d, x in zip(r.index, r)]
def build_recovery(hl="", view="pax"):
    if view == "pax":
        f = go.Figure([line(g, DATES, lst(paxf[g].loc[START:] / 1e6, 3), HV_PAX[g], True if g in big else "legendonly", True, hl) for g in show])
        f.update_layout(yaxis=dict(title="Passengers per month (millions)", rangemode="tozero"))
    else:
        f = go.Figure([line(g, DATES, lst(ratio[g].loc[START:] * 100, 1), HV_PCT[g], True, True, hl) for g in keep])
        f.add_trace(go.Scatter(x=DATES, y=[100] * len(DATES), mode="lines", line=dict(color="black", dash="dash", width=1), name="Same as 2019 (100%)", showlegend=False,
                               hovertemplate="Same as the same month of 2019<extra></extra>"))
        f.update_layout(yaxis=dict(title="% of the same month of 2019 (100 = same as 2019)", range=[0, 200]))
    f.update_layout(xaxis=dict(range=[DATES[0], X_END]), uirevision=view)       # uirevision keeps legend clicks/zoom when the airline is re-picked
    waves(f)
    lt = paxf["Go First"][paxf["Go First"] > 0].index.max()
    f.add_annotation(x=iso([lt])[0], y=0, text=f"Go First stops flying<br>({mon(lt)})", showarrow=True, arrowhead=2, ax=0, ay=-55, font=dict(size=11, color=DARK))
    return style(f, 500)

# ---- 3. month-by-month race: bars = this month, black tick = the same month of 2019 (the slider is a Dash control)
race = [g for g in show if paxf[g].loc["2020-01-01":].max() >= 40_000]
race.sort(key=lambda g: (-paxf[g].iloc[n3].mean(), -b19[g].sum()))
rm = paxf.loc["2020-01-01":].index
vmax = max(paxf[g].loc["2020-01-01":].max() for g in race) / 1e6
def build_race(i=0):
    d = rm[i]
    vals = [None if pd.isna(paxf[g].loc[d]) else paxf[g].loc[d] / 1e6 for g in race]       # None = the group had not started yet
    ticks = [(b19m[g].loc[d.month] / 1e6 if pd.notna(b19m[g].loc[d.month]) and b19m[g].loc[d.month] > 0 else None) for g in race]
    cols = [LIGHT if tk is None else (HL if (v or 0) >= tk else MID) for v, tk in zip(vals, ticks)]
    hov = [f"<b>{g}</b><br>{mon(d)}: {(v or 0):.2f} million passengers" + (f"<br>same month of 2019: {tk:.2f} million" if tk else "<br>no 2019 level to compare") for g, v, tk in zip(race, vals, ticks)]
    bar = go.Bar(x=vals, y=race, orientation="h", marker=dict(color=cols, line=dict(color=DARK, width=.6)), text=["" if v is None else f"{v:.1f} M" for v in vals], textposition="outside", hovertext=hov, hoverinfo="text", showlegend=False)
    tick = go.Scatter(x=ticks, y=race, mode="markers", marker=dict(symbol="line-ns", size=26, line=dict(color="black", width=3)), hoverinfo="skip", showlegend=False)
    dummies = [go.Bar(x=[None], y=[None], orientation="h", name=n, marker=dict(color=c, line=dict(color=DARK, width=.6)))
               for n, c in [("Above the same month of 2019", HL), ("Below the same month of 2019", MID), ("Started after 2019 (nothing to compare)", LIGHT)]]
    dummies.append(go.Scatter(x=[None], y=[None], mode="markers", name="Same month of 2019", marker=dict(symbol="line-ns", size=14, line=dict(color="black", width=3))))
    f = go.Figure([bar, tick] + dummies)
    f.update_layout(xaxis=dict(range=[0, vmax * 1.15], title="Passengers in the month (millions)"), yaxis=dict(autorange="reversed", automargin=True),
                    title=dict(text=f"<b>{pd.Timestamp(d):%B %Y}</b>", x=0.01, font=dict(size=22, color=DARK)), barmode="overlay", transition=dict(duration=0))
    style(f, 520); f.update_layout(legend=dict(orientation="h", y=-0.15, x=0), margin=dict(l=130, r=50, t=60, b=90))
    return f

# ---- 4. who carries the passengers
def build_share():
    order = [g for g in ["IndiGo", "Air India Group", "SpiceJet", "Go First", "Akasa Air"] if g in paxf.columns]
    shs = sh_all.loc[START:].where(total.loc[START:] > 0)               # no share in months with no flights at all
    other = 100 - shs[order].sum(axis=1)
    cols = [HL, DARK, MID, "#b5b5b5", "#d6d6d6"][:len(order)] + ["#ececec"]
    f = go.Figure()
    for g, c in zip(order + ["All other airlines"], cols):
        f.add_trace(go.Scatter(x=DATES, y=lst(shs[g] if g in order else other, 2), mode="lines", stackgroup="one", name=g, line=dict(width=.6, color="white"), fillcolor=c,
                               hovertemplate=f"{g}: %{{y:.0f}} of every 100 passengers<extra></extra>"))
    waves(f)
    f.update_layout(hovermode="x unified", yaxis=dict(title="Passengers (out of every 100)", range=[0, 100]), xaxis=dict(range=[DATES[0], DATES[-1]]))
    return style(f, 470)

# ---- 5. are the planes as full as before? (two views: 12-month average, or each month)
lfg = [g for g in keep + ["Akasa Air"] if g in pkp.columns]
mon_lf = (pkp / skp.where(skp > 0) * 100).loc[START:]
yr_lf = (pkp.rolling(12, min_periods=12).sum() / skp.rolling(12, min_periods=12).sum().where(lambda s: s > 0) * 100).loc[START:]
def build_lf(hl="", view="12"):
    tbl = yr_lf if view == "12" else mon_lf
    f = go.Figure([line(g, DATES, lst(tbl[g], 1), [f"<b>{g}</b><br>{mon(d)}: {v:.0f} of every 100 seats filled" + ("" if view == "1" else " (12-month average)") for d, v in tbl[g].items()], True, True, hl) for g in lfg])
    f.update_layout(xaxis=dict(range=[DATES[0], X_END]), yaxis=dict(title="Seats filled (out of every 100)", range=[30, 100]), uirevision=view)
    waves(f)
    return style(f, 470)

# ---- 6. scorecard: change in passengers per month (millions) or % of 2019
rows6 = []
for g in keep:
    now = paxf[g].iloc[n3].mean() / 1e6; b = np.mean([b19m[g].loc[d.month] for d in last3]) / 1e6
    rows6.append((g, now, b, now - b, now / b * 100))
rows6.sort(key=lambda r: -r[3]); names6 = [r[0] for r in rows6]
hov6 = [f"<b>{g}</b><br>{now:.2f} million passengers a month now<br>{b:.2f} million in the same months of 2019" for g, now, b, d, p in rows6]
def build_score(view="abs"):
    if view == "abs":
        f = go.Figure(go.Bar(x=[r[3] for r in rows6], y=names6, orientation="h", marker=dict(color=[HL if r[3] >= 0 else MID for r in rows6], line=dict(color=DARK, width=.6)),
                             text=[f"{r[3]:+.1f} M" for r in rows6], textposition="outside", cliponaxis=False, hovertext=hov6, hoverinfo="text", showlegend=False))
        f.update_layout(xaxis=dict(title="Passengers per month now minus the same months of 2019 (millions)", zeroline=True, zerolinewidth=2, zerolinecolor="black", automargin=True))
    else:
        f = go.Figure(go.Bar(x=[r[4] for r in rows6], y=names6, orientation="h", marker=dict(color=[HL if r[4] >= 100 else MID for r in rows6], line=dict(color=DARK, width=.6)),
                             text=[f"{r[4]:.0f}%" for r in rows6], textposition="outside", cliponaxis=False, hovertext=hov6, hoverinfo="text", showlegend=False))
        f.add_vline(x=100, line_dash="dash", line_color="black", line_width=1)
        f.update_layout(xaxis=dict(title="% of the same months of 2019 (100 = same as 2019)", range=[0, 175], automargin=True))
    f.update_layout(yaxis=dict(autorange="reversed", automargin=True))
    style(f, 380, legend=False); f.update_layout(margin=dict(l=130, r=60, t=40, b=60))
    return f

# ---- 7. before vs after cleaning
core = ["Aircraft Number", "Aircraft Hours", "Aircraft Kilometres", "Passenger Number", "Passenger Kilometers", "Seat Kilometers", "Passenger Load Factor"]
# missing values at each step of cleaning, so the change in population is not mistaken for an improvement in quality
METRICS16 = ["Aircraft Number", "Aircraft Hours", "Aircraft Kilometres", "Passenger Number", "Passenger Kilometers", "Seat Kilometers", "Passenger Load Factor", "Freight", "Mail",
             "Total Cargo", "Passenger Tonne Kilometer", "Mail Tonne Kilometer", "Freight Tonne Kilometer", "Total Tonne Kilometer", "Available Tonne Kilometer", "Weight Load Factor"]
st0 = interim
st1 = st0[~st0[METRICS16].isna().all(axis=1)]                                           # empty template rows removed
st2 = st1[st1.Type == "ScheduledDomestic"]                                               # the other sections (international, non-scheduled) set aside
st3 = st2[~st2.Airline.astype(str).str.casefold().str.replace(r"[_-]+", " ", regex=True).str.strip().eq("total domestic")]     # DGCA's aggregate rows set aside
STAGES = [("1. All parsed rows", st0), ("2. Empty template rows removed", st1), ("3. Scheduled domestic only", st2), ("4. Aggregate rows removed (final)", st3)]
na_stage = [(n, len(d), d[core].isna().mean() * 100) for n, d in STAGES]
na_b, na_a = na_stage[0][2], na_stage[1][2]                      # same rows, with and without the empty templates
print("cleaned table rows:", len(clean), "| rebuilt from the interim table:", len(st3))
brands = ["Air India", "Air India Express", "Vistara", "AirAsia India", "AIX Connect"]
sdf = clean[clean.Type == "ScheduledDomestic"]
bp = sdf[sdf.Airline.isin(brands)].groupby(["Airline", "Date"])["Passenger Number"].sum().unstack("Airline") / 1e6
gp = sdf[sdf.Group == "Air India Group"].groupby("Date")["Passenger Number"].sum() / 1e6
def build_quality():
    x7 = pd.Timestamp("2022-01-01")
    f = make_subplots(rows=1, cols=2, column_widths=[.42, .58], subplot_titles=("Empty cells per column, at each step of cleaning (% of rows)", "Air India Group: brands separate vs combined"), horizontal_spacing=.14)
    for (n, rows, na), col in zip(na_stage, ["#d9d9d9", MID, "#fdae6b", HL]):
        f.add_trace(go.Bar(y=core, x=na.values, orientation="h", name=f"{n} ({rows:,} rows)", marker=dict(color=col), hovertemplate="%{y}: %{x:.1f}% empty<extra>" + n + "</extra>", legendgroup="stages"), 1, 1)
    for c in brands:
        s = bp[c].loc[x7:LAST].dropna()
        f.add_trace(go.Scatter(x=iso(s.index), y=lst(s, 3), mode="lines", name=c, line=dict(color=MID, width=1.4), hovertemplate=f"{c}: %{{y:.2f}} million<extra></extra>", legendgroup="brands"), 1, 2)
    s = gp.loc[x7:LAST]
    f.add_trace(go.Scatter(x=iso(s.index), y=lst(s, 3), mode="lines", name="Air India Group (combined)", line=dict(color=HL, width=4), hovertemplate="Air India Group: %{y:.2f} million<extra></extra>"), 1, 2)
    f.update_yaxes(autorange="reversed", row=1, col=1); f.update_yaxes(title_text="Passengers per month (millions)", row=1, col=2)
    f.update_xaxes(title_text="Empty cells (% of rows)", row=1, col=1)
    f.update_layout(barmode="group"); style(f, 520); f.update_layout(margin=dict(l=150, r=40, t=60, b=120), legend=dict(orientation="h", y=-0.22))
    return f
# ---- 8. capacity growth: seat-kilometres against the same month of 2019 (two views)
def build_capacity(hl="", view="line"):
    if view == "line":
        f = go.Figure([line(g, DATES, lst(cap_ratio[g].loc[START:], 1),
                            [f"<b>{g}</b><br>{mon(d)}: capacity {x:.0f}% of {pd.Timestamp(d):%B} 2019<br>({paxf[g].loc[d] / sk_pax_den[g].loc[d.month] * 100:.0f}% as many passengers)" for d, x in cap_ratio[g].loc[START:].items()],
                            True, True, hl) for g in cap_ok])
        f.add_trace(go.Scatter(x=DATES, y=[100] * len(DATES), mode="lines", line=dict(color="black", dash="dash", width=1), showlegend=False, hovertemplate="Same as the same month of 2019<extra></extra>"))
        f.update_layout(xaxis=dict(range=[DATES[0], X_END]), yaxis=dict(title="Seat-kilometres, % of the same month of 2019 (100 = same as 2019)", range=[0, 250]), uirevision=view)
        waves(f)
        return style(f, 470)
    names = [r[0] for r in CAP_ROWS]
    f = go.Figure([go.Bar(y=names, x=[r[1] for r in CAP_ROWS], orientation="h", name="Capacity (seat-kilometres)", marker=dict(color=HL, line=dict(color=DARK, width=.6)), text=[f"{r[1]:.0f}%" for r in CAP_ROWS], textposition="outside", cliponaxis=False,
                          hovertemplate="<b>%{y}</b><br>capacity %{x:.0f}% of 2019<extra></extra>"),
                   go.Bar(y=names, x=[r[2] for r in CAP_ROWS], orientation="h", name="Passengers", marker=dict(color=MID, line=dict(color=DARK, width=.6)), text=[f"{r[2]:.0f}%" for r in CAP_ROWS], textposition="outside", cliponaxis=False,
                          hovertemplate="<b>%{y}</b><br>passengers %{x:.0f}% of 2019<extra></extra>")])
    f.add_vline(x=100, line_dash="dash", line_color="black", line_width=1)
    f.update_layout(barmode="group", xaxis=dict(title="% of the same months of 2019 (100 = same as 2019)", range=[0, max(r[1] for r in CAP_ROWS + [(0, 0, 0)]) * 1.15 + 10], automargin=True), yaxis=dict(autorange="reversed", automargin=True))
    style(f, 420); f.update_layout(margin=dict(l=130, r=60, t=40, b=90), legend=dict(orientation="h", y=-0.2, x=0))
    return f
sk_pax_den = b19m                                                         # passengers in the same month of 2019 (rows: month number)
CAP_ROWS = [(g, cap_ratio[g].iloc[n3].mean(), ratio[g].iloc[n3].mean() * 100) for g in cap_ok]
CAP_ROWS.sort(key=lambda r: -r[1])

import itertools
# ---- 9. route density: the six metros joined by curved routes on a map of India, one month at a time (the slider and Play button are Dash controls)
ARC_PTS = np.linspace(0, 1, 24)
def arc(a, b, bend=0.2):
    """Curved path between two cities: a quadratic curve bent to one side. Returns lons, lats and the point halfway along (where the plane sits)."""
    (la, oa), (lb, ob) = COORDS[a], COORDS[b]
    dy, dx = lb - la, ob - oa
    cy, cx = (la + lb) / 2 - bend * dx, (oa + ob) / 2 + bend * dy       # control point, pushed to the side
    t = ARC_PTS
    lat = (1 - t) ** 2 * la + 2 * t * (1 - t) * cy + t ** 2 * lb
    lon = (1 - t) ** 2 * oa + 2 * t * (1 - t) * cx + t ** 2 * ob
    return list(lon), list(lat), (0.25 * oa + 0.5 * cx + 0.25 * ob, 0.25 * la + 0.5 * cy + 0.25 * lb)
ARCS = {(a, b): arc(a, b) for a, b in itertools.combinations(sorted(MAJOR), 2)}
# traffic bands are fixed shares of the busiest pair ever, so a thicker, darker line always means more passengers, in every month
BAND_EDGES = [0, .04, .1, .2, .35, .55, .8, 1.01]
BAND_W = [1.5, 3, 5, 8, 12, 17, 23]
BAND_COL = ["#fdd0a2", "#fdae6b", "#fd8d3c", "#f16913", "#d94801", "#a63603", "#7f2704"]
def build_route_map(i=0):
    d = RT_MONTHS[i]; pm = pairs_by_month[d]; pm = pm[pm.pax > 0]; traces = []
    band = pd.cut(pm.pax / PAIR_MAX, BAND_EDGES, labels=False, right=False)
    for k in range(len(BAND_W)):
        xs, ys = [], []
        for r in pm[band == k].itertuples():
            lo, la, _ = ARCS[(r.A, r.B)]; xs += lo + [None]; ys += la + [None]
        traces.append(go.Scattergeo(lon=xs, lat=ys, mode="lines", line=dict(width=BAND_W[k], color=BAND_COL[k]), hoverinfo="skip", showlegend=False))
    mids = [ARCS[(r.A, r.B)][2] for r in pm.itertuples()]
    traces.append(go.Scattergeo(lon=[m[0] for m in mids], lat=[m[1] for m in mids], mode="markers",
                                marker=dict(size=22, opacity=0), hoverinfo="text", showlegend=False,
                                hovertext=[f"<b>{r.A.title()} – {r.B.title()}</b><br>{r.pax / 1e3:,.0f} thousand passengers (both directions)" for r in pm.itertuples()]))
    traces.append(go.Scattergeo(lon=[m[0] for m in mids], lat=[m[1] for m in mids], mode="text", text=[f"{r.pax / 1e3:,.0f}k" for r in pm.itertuples()], textposition="bottom center",
                                textfont=dict(size=11, color="#7f2704"), hoverinfo="skip", showlegend=False))
    cv = cvol.loc[d, MAJOR]
    traces.append(go.Scattergeo(lon=[COORDS[c][1] for c in MAJOR], lat=[COORDS[c][0] for c in MAJOR], mode="markers+text", text=[c.title() for c in MAJOR], textposition="top center", textfont=dict(size=13, color=DARK),
                                marker=dict(size=[10 + 38 * (v / CITY_MAX) for v in cv], color=DARK, opacity=.85, line=dict(color="white", width=1.5)), hoverinfo="text", showlegend=False,
                                hovertext=[f"<b>{c.title()}</b><br>{v / 1e6:.2f} million passengers arriving and leaving (all routes)" for c, v in zip(MAJOR, cv)]))
    f = go.Figure(traces)
    if d in RT_MISSING:
        f.add_annotation(x=0.5, y=0.5, xref="paper", yref="paper", showarrow=False, align="center", font=dict(size=17, color=DARK), bgcolor="rgba(255,255,255,.9)", bordercolor=MID, borderpad=10,
                         text=f"<b>No route data for {pd.Timestamp(d):%B %Y}</b><br>DGCA's city-pair file for this month is not available,<br>so there are no routes to draw. Move the slider to another month.")
    f.update_geos(scope="asia", resolution=50, lonaxis_range=[68, 93], lataxis_range=[8, 32], showcountries=True, countrycolor="#bdbdbd", showland=True, landcolor="#f4f4f2",
                  showocean=True, oceancolor="#eaf1f7", showcoastlines=True, coastlinecolor="#bdbdbd", showframe=False)
    f.update_layout(template="plotly_white", font=dict(family=FONT, size=13, color="#222"), height=640, margin=dict(l=0, r=0, t=50, b=0), showlegend=False,
                    title=dict(text=f"<b>{pd.Timestamp(d):%B %Y}</b>" + ("" if d in RT_MISSING else f" · {pm.pax.sum() / 1e6:.2f} million passengers on these {N_ROUTES} routes"), x=0.01, font=dict(size=20, color=DARK)))
    return f
print("figures ready | race months:", len(rm), "| route months:", len(RT_MONTHS))

## 4. Captions and page text

In [ ]:
# 4. CAPTIONS (numbers computed from the data) AND PAGE TEXT
S = stats
meta = json.load(open(latest(INTERIM, "collection_metadata_*.json")))
last_rm = rm[-1]
race_above = [g for g in race if g in keep and paxf[g].loc[last_rm] >= b19m[g].loc[last_rm.month]]
race_cmp = [g for g in race if g in keep]
gain = sum(r[3] for r in rows6 if r[3] > 0); loss = -sum(r[3] for r in rows6 if r[3] < 0)
ask_chg = (skp["SpiceJet"].iloc[-12:].sum() / skp["SpiceJet"].loc["2019-01-01":"2019-12-01"].sum() - 1) * 100
zero_rows = int((clean["Passenger Number"].eq(0) & clean["Passenger Kilometers"].eq(0) & clean["Seat Kilometers"].eq(0) & clean["Passenger Load Factor"].isna()).sum())
o24, d24 = pd.Timestamp("2024-10-01"), pd.Timestamp("2024-12-01")
sg = lambda x: f"{x:+.1f}".replace("-", "−")
CR = {g: (None, c, p) for g, c, p in CAP_ROWS}
pax_total_now = total.iloc[n3].mean() / tot19_by_month.reindex(total.index[n3].month).mean() * 100
bp_last = pairs_by_month[RT_MONTHS[-1]].nlargest(1, "pax").iloc[0]
top_share = pairs_by_month[RT_MONTHS[-1]].pax.sum() / pairs[pairs.Date == RT_MONTHS[-1]].pax.sum() * 100
pm19 = pairs_by_month[pd.Timestamp("2019-01-01")].pax.sum()
rt_airline_note = "DGCA publishes city-pair passengers for all airlines together, so the map shows the whole market."
CAP = {
 "market": f"In 2019 about {avg19:.1f} million people a month took a scheduled domestic flight. In {mon(trough_date)} that fell to {"zero" if trough_val < 0.05 else f"{trough_val:.1f} million"}, and the market first matched its 2019 level again in {fm(market_back)}. "
           f"Over the last 12 months it averaged {l12:.1f} million a month, {l12 / avg19 * 100:.0f}% of 2019.",
 "recovery": f"{HL_GROUP} now carries {S[HL_GROUP]['now_m']:.1f} million passengers a month against {S[HL_GROUP]['avg19_m']:.1f} million in 2019 ({S[HL_GROUP]['pct']}%), and Air India Group {S['Air India Group']['now_m']:.1f} million against {S['Air India Group']['avg19_m']:.1f} million ({S['Air India Group']['pct']}%). "
             f"SpiceJet is at {S['SpiceJet']['pct']}% of 2019 and Go First stopped flying after {fm(paxf['Go First'][paxf['Go First'] > 0].index.max())}.",
 "race": f"In {last_rm:%B %Y}, {len(race_above)} of the {len(race_cmp)} groups that flew in the same month of 2019 are above their black tick ({', '.join(race_above)}). "
         f"Go First's bar vanishes after May 2023. Akasa Air, which began in August 2022, has no 2019 tick, and Star Air's tick is its launch year, so neither is counted.",
 "share": f"{top} carried {share19.get(top, 0) * 100:.0f} of every 100 passengers in 2019 and {sh_all[top].iloc[n3].mean():.0f} of every 100 in the latest three months. "
          f"Air India Group moved from {S['Air India Group']['share19']:.0f} to {S['Air India Group']['share_now']:.0f}, SpiceJet from {S['SpiceJet']['share19']:.0f} to {S['SpiceJet']['share_now']:.0f}, and Akasa Air went from 0 to {S['Akasa Air']['share_now']:.0f}.",
 "lf": f"Planes are about as full as before, a little less for IndiGo. Over the last 12 months {HL_GROUP} filled {S[HL_GROUP]['lf_now']} of every 100 seats ({S[HL_GROUP]['lf19']} in 2019) and Air India Group {S['Air India Group']['lf_now']} ({S['Air India Group']['lf19']}). "
       f"SpiceJet filled {S['SpiceJet']['lf_now']} ({S['SpiceJet']['lf19']} in 2019): somewhat emptier planes, but the bigger story is that it now flies {abs(ask_chg):.0f}% fewer seat-kilometres than in 2019.",
 "score": f"The groups that grew added {gain:.1f} million passengers a month compared with the same months of 2019, led by {rows6[0][0]} ({sg(rows6[0][3])} million). "
          f"The groups that shrank or stopped lost {loss:.1f} million a month, the most by {rows6[-1][0]} ({sg(rows6[-1][3])} million). Akasa Air (about {S['Akasa Air']['now_m']:.1f} million a month, started 2022) is not shown because it has no 2019 level.",
 "capacity": f"Across all airlines the seats on sale (seat-kilometres) are now {cap_total_now:.0f}% of the same months of 2019, while passengers are {pax_total_now:.0f}%. "
             f"IndiGo has {CR['IndiGo'][1]:.0f}% of its 2019 capacity and {CR['IndiGo'][2]:.0f}% of its passengers; Air India Group {CR['Air India Group'][1]:.0f}% and {CR['Air India Group'][2]:.0f}%; SpiceJet {CR['SpiceJet'][1]:.0f}% and {CR['SpiceJet'][2]:.0f}%.",
 "route": f"In {fm(RT_MONTHS[-1])} the busiest pair was {bp_last.A.title()} and {bp_last.B.title()} with {bp_last.pax / 1e3:,.0f} thousand passengers. The {N_ROUTES} routes between these {N_CITIES} cities carried {top_share:.0f} of every 100 passengers in the country, {pairs_by_month[RT_MONTHS[-1]].pax.sum() / pm19 * 100:.0f}% of their January 2019 traffic. "
          f"In {mon(trough_date)}, when almost nothing flew, the same map is nearly empty. {rt_airline_note}",
 "quality": f"Left: removing the empty template rows (same rows, step 1 to 2) cuts the empty share of Passenger Number from {na_b['Passenger Number']:.1f}% to {na_a['Passenger Number']:.1f}%. Steps 3 and 4 only change which rows are counted (scheduled domestic, no aggregate rows), so they are a change of population, not a repair. {zero_rows} of the {int(clean['Passenger Load Factor'].isna().sum())} empty load factors are months with zero flying, where 0/0 is undefined. "
            f"Right: after Vistara's last month the Air India brand alone jumps from {bp['Air India'].get(o24, np.nan):.2f} to {bp['Air India'].get(d24, np.nan):.2f} million, while the combined group stays continuous ({gp.get(o24, np.nan):.2f} to {gp.get(d24, np.nan):.2f} million).",
}
HOW = {
 "market": "Each point is one month. The dashed line is the 2019 monthly average, and the grey bands mark the lockdown and the second wave. Use the period buttons or the slider under the chart to zoom into a period.",
 "recovery": "Each line is one airline group. Choose an airline in the bar at the top to make it orange. Use the buttons to switch between real passenger numbers and a comparison with the same month of 2019, and click names in the legend to add or remove groups.",
 "race": "Press play, or drag the slider. Each bar is how many passengers the group carried that month; the black tick is what it carried in the same month of 2019. Orange means it has passed that tick.",
 "share": "Read it as 100 passengers. The orange layer at the bottom is the share that flies IndiGo. Hover to see the split for any month.",
 "lf": "A load factor is the share of seats that were filled. Use the buttons to switch between each month and a 12-month average. Choose an airline at the top to highlight it.",
 "score": "One bar per group that flew all of 2019 and was big enough to compare. The comparison uses the latest three months against the same three months of 2019.",
 "capacity": "Capacity is the seats an airline puts on sale, measured as seat-kilometres (seats multiplied by distance flown). Choose an airline at the top to highlight it. The second view sets capacity next to passengers for the latest three months.",
 "route": "Each curved line joins two of the six biggest metros. A thicker, darker line means more passengers that month, and the number on it is the passengers on that route in thousands. Dot size shows how many passengers arrive in and leave each city. Press play or drag the slider to move through time. DGCA reports city pairs for all airlines together, so the map shows the whole market.",
 "quality": "The same data before and after cleaning. Hover over the bars and lines for the exact values.",
}
TITLES = {"market": "How many people fly?", "recovery": "Airline by airline: who is back?", "race": "Watch it month by month",
          "share": "Who carries the passengers?", "lf": "Are the planes as full as before?", "score": "Scorecard: who gained and who lost?", "capacity": "Are airlines flying more seats than before?", "route": "Route density: where do people fly?", "quality": "Data quality: before and after cleaning"}
TAB_LABELS = {"market": "Total passengers", "recovery": "Airline by airline", "race": "Month by month", "share": "Market share", "lf": "Load factor", "capacity": "Capacity growth",
              "score": "Scorecard", "route": "Route density", "quality": "Data quality", "method": "Methodology and sources"}
excl_txt = "; ".join(f"{g} ({why})" for g, why in excluded.items())
print(CAP["market"])

## 5. The Dash app: layout and callbacks

In [ ]:
# 5. THE DASH APP: layout (header cards, sticky tabs + airline picker, one panel per tab) and callbacks
CSS = """
:root{--hl:#d95f02;--dark:#404040;--mid:#8c8c8c;--bg:#f4f4f2}
*{box-sizing:border-box}body{margin:0;font-family:Arial,Helvetica,sans-serif;background:var(--bg);color:#222;line-height:1.5}
.wrap{max-width:1120px;margin:0 auto;padding:0 18px}
header{background:#fff;border-bottom:1px solid #ddd;padding:26px 0 10px}
h1{font-size:clamp(1.45rem,3.2vw,2.1rem);margin:0 0 6px;line-height:1.2}.sub{color:#555;margin:0 0 4px}
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(230px,1fr));gap:14px;margin:20px 0 10px}
.card{background:#fff;border:1px solid #e0e0e0;border-radius:10px;padding:14px 16px}.card .big{font-size:1.85rem;font-weight:700;color:var(--hl);line-height:1.15}
.card .lab{font-weight:600}.card .small{color:#666;font-size:.88rem}
#nav{position:sticky;top:0;z-index:30;background:#fff;border-bottom:1px solid #ccc;box-shadow:0 2px 6px rgba(0,0,0,.06)}
#nav .tab-parent{overflow-x:auto}
.tabs{display:flex;gap:4px;white-space:nowrap}
.tab{flex:0 0 auto;white-space:nowrap;font:inherit;font-size:.95rem;background:none;border:0;border-bottom:3px solid transparent;padding:11px 14px;color:#444;cursor:pointer}
.tab:hover{background:#f4f4f2}.tab--selected{color:#111;font-weight:700;border-bottom-color:var(--hl)}
#picker{padding:10px 0;border-top:1px solid #e6e6e6}
#picker .row{display:flex;flex-wrap:wrap;gap:10px 18px;align-items:center}#picker .dd{width:260px}
#picker .sentence{font-weight:600;flex:1 1 360px}#picker .mini{color:#444;font-size:.9rem;gap:6px 22px}#picker .mini b{color:var(--hl)}
section.panel{background:#fff;border:1px solid #e0e0e0;border-radius:10px;padding:16px 20px 18px;margin:20px 0}
h2{margin:0 0 2px;font-size:1.3rem}.how{color:#555;font-size:.93rem;margin:.1rem 0 .6rem}
.controls{display:flex;flex-wrap:wrap;align-items:center;gap:10px 16px;margin:.3rem 0 .4rem}
.controls label{margin-right:14px;cursor:pointer;font-size:.95rem}.controls input{margin-right:5px;accent-color:var(--hl)}
.btn{font:inherit;padding:6px 16px;border:1px solid #999;border-radius:6px;background:#fff;cursor:pointer}.btn:hover{background:#f4f4f2}
.race-slider{flex:1 1 420px;min-width:260px}
.cap{border-left:4px solid var(--hl);background:#fafafa;padding:.45rem .8rem;margin:.8rem 0 0}
.tabpanel h3{margin:1.2rem 0 .3rem;font-size:1.05rem}.tabpanel ul{margin:.3rem 0 .6rem;padding-left:1.2rem}.tabpanel li{margin:.3rem 0}
footer{color:#555;font-size:.88rem;padding:10px 0 40px}a{color:#8a3f00}
"""
TAB_STYLE = {"border": "0", "borderBottom": "3px solid transparent", "background": "none", "padding": "11px 14px", "color": "#444"}
TAB_SELECTED = {**TAB_STYLE, "borderBottom": "3px solid #d95f02", "color": "#111", "fontWeight": "700"}
app = Dash(__name__, title="Indian airlines after COVID-19: who got their passengers back?", suppress_callback_exceptions=True)
app.index_string = app.index_string.replace("</head>", f"<style>{CSS}</style></head>")

def radio(cid, options, value):
    return dcc.RadioItems(id=cid, options=[{"label": l, "value": v} for l, v in options], value=value, className="controls", inline=True)
def panel(k, controls, graph):
    return html.Section(className="panel tabpanel", children=[
        html.H2(TITLES[k]), html.P(HOW[k], className="how"), controls, graph,
        html.P([html.B("What it shows. "), CAP[k]], className="cap")])
def graph(gid, fig): return dcc.Graph(id=gid, figure=fig, config=CFG, style={"width": "100%"})

race_marks = {i: mon(d) for i, d in enumerate(rm) if d.month == 1}
def method_panel():
    li = lambda *c: html.Li(list(c))
    return html.Section(className="panel tabpanel", children=[
        html.H2("Methodology and sources"),
        html.H3("Data and coverage"),
        html.Ul([li("Scheduled domestic flights, monthly, from January 2019 to ", f"{LAST:%B %Y}", "."),
                 li(html.B(f"Data ends {LAST:%B %Y}."), " The next month is left out because the airlines do not add up to DGCA's own total yet. DGCA labels its figures provisional.")]),
        html.H3("How we counted"),
        html.Ul([li(html.B("Back to 2019"), " means the average of three months reached the average of the same three months of 2019, so seasons do not distort the comparison. The current position uses the latest three months."),
                 li(html.B("Airline groups."), " Air India, Air India Express, Vistara, AirAsia India and AIX Connect are combined as Air India Group for all years, and JetLite is counted with Jet Airways, so mergers do not look like collapses."),
                 li(html.B("Stopped airlines"), " count as zero passengers after their last month with traffic."),
                 li(html.B("Fair comparison only."), f" A group needs passengers in all 12 months of 2019 and at least 0.5% of that year's passengers. Not compared: {excl_txt}."),
                 li(html.B("Load factor"), " is the share of seats that were filled. The load factor chart shows each month and a 12-month average."),
                 li(html.B("Capacity"), " is seat-kilometres: the seats an airline puts on sale multiplied by the distance flown. It is compared with the same month of 2019 in the same way as passengers."),
                 li(html.B("Scheduled flights only."), " Non-scheduled domestic flying (charters and similar) is removed from the data in the cleaning step and appears nowhere in this dashboard."),
                 li(html.B("Route density."), f" City-pair passengers (both directions added together) come from DGCA's monthly city-pair files. They cover all airlines together and end in {fm(RT_MONTHS[-1])}. {'Months the Vonter table lacks (' + ', '.join(SUPPLEMENT_MONTHS) + ") come from DGCA's own city-pair files, because the Vonter table lacks them (the download notebook fetches them)." if SUPPLEMENT_MONTHS else ''} {'No city-pair data exists for ' + ', '.join(fm(d) for d in RT_MISSING) + ', so the map has no routes for those months.' if RT_MISSING else ''} The map shows the six biggest metros (Delhi, Mumbai, Bengaluru, Hyderabad, Kolkata, Chennai) and the {N_ROUTES} routes between them. Line colour and thickness use fixed bands, so they mean the same thing in every month. The city-pair totals match the airline totals to within {abs(1 - chk.min()) * 100:.1f}%. Goa's two airports are shown as one city, and Navi Mumbai is counted with Mumbai.")]),
        html.H3("Data cleaning"),
        html.Ul([li(f"Removing the empty template rows cuts the empty share of Passenger Number from {na_b['Passenger Number']:.1f}% to {na_a['Passenger Number']:.1f}% (same rows, before and after). The later steps change which rows are counted, so the quality chart shows every step separately."),
                 li(f"A month with no report counts as zero passengers only where there is evidence of no flying: DGCA reported zero, nobody reported anything that month, or the airlines still add up to DGCA's Total Domestic without it. {len(unknown_months)} missing group-months since 2019 have no such evidence and are left unknown instead of zero."),
                 li("Recovery: the cards and captions give the first month the 3-month average reached 100% of 2019 and, where it differs, the month from which it stayed at or above 100% for at least six months."),
                 li(f"{zero_rows} of the {int(clean['Passenger Load Factor'].isna().sum())} empty load factors are months with zero flying, where 0/0 is undefined."),
                 li(f"After Vistara's last month the Air India brand alone jumps from {bp['Air India'].get(o24, np.nan):.2f} to {bp['Air India'].get(d24, np.nan):.2f} million, while the combined group stays continuous ({gp.get(o24, np.nan):.2f} to {gp.get(d24, np.nan):.2f} million). This is why the airline groups above are used.")]),
        html.H3("Sources"),
        html.Ul([li("DGCA monthly traffic statistics (Excel files), ", html.A("dgca.gov.in", href="https://www.dgca.gov.in/digigov-portal/?baseLocale=en_US?page=259/4184/servicename"), f", collected {meta['collection_date']}."),
                 li("Second copy for cross-checking: ", html.A("Vonter/india-aviation-traffic", href="https://github.com/Vonter/india-aviation-traffic"), f" (commit {meta['github_commit'][:7]}), the same DGCA files parsed separately."),
                 li("City-pair passengers: DGCA data parsed by ", html.A("Vonter/india-aviation-traffic", href="https://github.com/Vonter/india-aviation-traffic"), f" (commit {meta['github_commit'][:7]}), released under the Open Database License. Source: DGCA and the Ministry of Civil Aviation."),
                 li("Charts: Plotly and Dash. Data handling: pandas."),
                 li("Licences, terms of use and the AI tool used are listed in the project report.")])])

def player(prefix, n, marks):
    return html.Div(className="controls", children=[
        html.Button("▶ Play", id=f"{prefix}-play", className="btn", n_clicks=0),
        html.Div(dcc.Slider(id=f"{prefix}-slider", min=0, max=n - 1, step=1, value=0, marks=marks, updatemode="drag", tooltip={"always_visible": False}), className="race-slider"),
        dcc.Interval(id=f"{prefix}-tick", interval=450, disabled=True)])
rt_marks = {i: mon(d) for i, d in enumerate(RT_MONTHS) if d.month == 1}

def tab_content(tab):
    if tab == "market":
        return panel(tab, radio("market-range", [("All years", "all"), ("2019 to 2021", "2019-21"), ("2022 onwards", "2022+")], "all"), graph("fig-market", build_market()))
    if tab == "recovery":
        return panel(tab, radio("rec-view", [("Passengers per month", "pax"), ("Compared with the same month of 2019", "pct")], "pax"), graph("fig-recovery", build_recovery(HL_GROUP)))
    if tab == "race":
        return panel(tab, player("race", len(rm), race_marks), graph("fig-race", build_race(0)))
    if tab == "share": return panel(tab, None, graph("fig-share", build_share()))
    if tab == "lf":
        return panel(tab, radio("lf-view", [("12-month average (smoother)", "12"), ("Each month", "1")], "12"), graph("fig-lf", build_lf(HL_GROUP)))
    if tab == "score":
        return panel(tab, radio("score-view", [("Passengers gained or lost per month", "abs"), ("% of what it carried in 2019", "pct")], "abs"), graph("fig-score", build_score()))
    if tab == "capacity":
        return panel(tab, radio("cap-view", [("Capacity compared with the same month of 2019", "line"), ("Capacity next to passengers (latest 3 months)", "bars")], "line"), graph("fig-capacity", build_capacity(HL_GROUP)))
    if tab == "route":
        return panel(tab, player("rd", len(RT_MONTHS), rt_marks), graph("fig-route", build_route_map(0)))
    if tab == "quality": return panel(tab, None, graph("fig-quality", build_quality()))
    return method_panel()

app.layout = html.Div([
    html.Header(html.Div(className="wrap", children=[
        html.H1("Which Indian airlines got their passengers back after COVID-19?"),
        html.P(f"Scheduled domestic flights, monthly data from the Directorate General of Civil Aviation (DGCA), January 2019 to {LAST:%B %Y}.", className="sub"),
        html.Div(className="cards", children=[html.Div(className="card", children=[html.Div(a, className="big"), html.Div(b, className="lab"), html.Div(c, className="small")]) for a, b, c in CARDS])])),
    html.Div(id="nav", children=[
        html.Div(className="wrap", children=dcc.Tabs(id="tabs", value="market", className="tabs", parent_className="tab-parent", style={"border": "0"},
                 children=[dcc.Tab(label=l, value=k, className="tab", selected_className="tab--selected", style=TAB_STYLE, selected_style=TAB_SELECTED) for k, l in TAB_LABELS.items()])),
        html.Div(id="picker", style={"display": "none"}, children=html.Div(className="wrap", children=[
            html.Div(className="row", children=[html.Label(html.B("Pick an airline group:"), htmlFor="pick"),
                dcc.Dropdown(id="pick", options=[{"label": "None (all groups equal)", "value": ""}] + [{"label": g, "value": g} for g in show], value=HL_GROUP, clearable=False, className="dd"),
                html.Span(id="pk-sentence", className="sentence")]),
            html.Div(id="pk-mini", className="row mini")]))]),
    html.Main(id="content", className="wrap"),
])

@app.callback(Output("content", "children"), Output("picker", "style"), Input("tabs", "value"))
def show_tab(tab):
    return tab_content(tab), {"display": "block" if tab in ("recovery", "lf", "capacity") else "none"}

@app.callback(Output("pk-sentence", "children"), Output("pk-mini", "children"), Input("pick", "value"))
def picker_text(name):
    s = stats.get(name)
    if not s: return "Showing every group in the same grey.", None
    na = lambda v, suf="": "n/a" if v is None else f"{v}{suf}"
    return f"{name}: {s['sentence']}", [
        html.Span(["Passengers a month now: ", html.B(f"{s['now_m']:.2f} M"), f" (2019: {s['avg19_m']:.2f} M)" if s["avg19_m"] is not None else ""]),
        html.Span(["Share of all passengers: ", html.B(f"{s['share_now']} in 100"), f" (2019: {s['share19']})"]),
        html.Span(["Seats filled: ", html.B(na(s["lf_now"], " in 100")), f" (2019: {na(s['lf19'])})"])]

@app.callback(Output("fig-market", "figure"), Input("market-range", "value"))
def _market(rng): return build_market(rng)
@app.callback(Output("fig-recovery", "figure"), Input("pick", "value"), Input("rec-view", "value"))
def _recovery(hl, view): return build_recovery(hl, view)
@app.callback(Output("fig-lf", "figure"), Input("pick", "value"), Input("lf-view", "value"))
def _lf(hl, view): return build_lf(hl, view)
@app.callback(Output("fig-capacity", "figure"), Input("pick", "value"), Input("cap-view", "value"))
def _capacity(hl, view): return build_capacity(hl, view)
@app.callback(Output("fig-route", "figure"), Input("rd-slider", "value"))
def _route(i): return build_route_map(i)
@app.callback(Output("fig-score", "figure"), Input("score-view", "value"))
def _score(view): return build_score(view)
@app.callback(Output("fig-race", "figure"), Input("race-slider", "value"))
def _race(i): return build_race(i)

def register_player(prefix, n):
    @app.callback(Output(f"{prefix}-slider", "value"), Output(f"{prefix}-tick", "disabled"), Output(f"{prefix}-play", "children"),
                  Input(f"{prefix}-play", "n_clicks"), Input(f"{prefix}-tick", "n_intervals"), State(f"{prefix}-slider", "value"), State(f"{prefix}-tick", "disabled"), prevent_initial_call=True)
    def play(_, __, i, paused):
        if ctx.triggered_id == f"{prefix}-play":                  # Play / Pause button
            if paused: return (0 if i >= n - 1 else i), False, "❚❚ Pause"
            return i, True, "▶ Play"
        if i >= n - 1: return i, True, "▶ Play"                   # timer reached the last month: stop
        return i + 1, False, "❚❚ Pause"
register_player("race", len(rm)); register_player("rd", len(RT_MONTHS))

## 6. Run it
Starts the app on http://127.0.0.1:8050 (stop the cell to shut it down).

In [ ]:
# 6. RUN THE APP (opens at http://127.0.0.1:8050; stop the cell to shut it down)
app.run(host="127.0.0.1", port=8050, debug=False, jupyter_mode="external")